## Task 5 - Optimization of BERT/DistilBERT to perform POS Tagging and Chunking

In [1]:
import warnings
import os
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="seqeval")
import seqeval
warnings.filterwarnings("ignore", module=r"seqeval\..*")

In [2]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

In [3]:
# Install libraries
%pip install -q transformers datasets seqeval evaluate accelerate --quiet
%pip install -q nltk --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import numpy as np
import torch
import nltk
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")
print(f"✅ PyTorch: {torch.__version__}")

✅ Device: cpu
✅ PyTorch: 2.11.0+cpu


In [13]:
# Sample dataset (tokens + POS + chunk tags)

data = [
    {
        "tokens": ["John", "works", "at", "Google"],
        "pos_tags": ["NNP", "VBZ", "IN", "NNP"],
        "chunk_tags": ["B-NP", "B-VP", "B-PP", "B-NP"],
    },
    {
        "tokens": ["She", "loves", "machine", "learning"],
        "pos_tags": ["PRP", "VBZ", "NN", "NN"],
        "chunk_tags": ["B-NP", "B-VP", "I-NP", "I-NP"],
    },
    {
        "tokens": ["I", "am", "learning", "NLP"],
        "pos_tags": ["PRP", "VBP", "VBG", "NNP"],
        "chunk_tags": ["B-NP", "B-VP", "I-VP", "B-NP"],
    },
]

In [14]:
dataset = Dataset.from_list(data)

dataset = DatasetDict({
    "train": dataset,
    "validation": dataset,
    "test": dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
})


In [15]:
# Get unique POS and chunk labels

pos_labels = list(set(label for example in data for label in example["pos_tags"]))
chunk_labels = list(set(label for example in data for label in example["chunk_tags"]))

print("POS Labels:", pos_labels)
print("Chunk Labels:", chunk_labels)

POS Labels: ['VBP', 'NNP', 'IN', 'NN', 'VBZ', 'PRP', 'VBG']
Chunk Labels: ['B-VP', 'I-VP', 'I-NP', 'B-PP', 'B-NP']


In [17]:
# Create mappings

pos_label2id = {label: i for i, label in enumerate(pos_labels)}
pos_id2label = {i: label for label, i in pos_label2id.items()}

chunk_label2id = {label: i for i, label in enumerate(chunk_labels)}
chunk_id2label = {i: label for label, i in chunk_label2id.items()}

print("POS mapping:", pos_label2id)
print("Chunk mapping:", chunk_label2id)

POS mapping: {'VBP': 0, 'NNP': 1, 'IN': 2, 'NN': 3, 'VBZ': 4, 'PRP': 5, 'VBG': 6}
Chunk mapping: {'B-VP': 0, 'I-VP': 1, 'I-NP': 2, 'B-PP': 3, 'B-NP': 4}


In [18]:
# Convert labels to numbers

def encode_labels(example):
    example["pos_tags"] = [pos_label2id[label] for label in example["pos_tags"]]
    example["chunk_tags"] = [chunk_label2id[label] for label in example["chunk_tags"]]
    return example

dataset = dataset.map(encode_labels)

dataset["train"][0]

Map: 100%|██████████| 3/3 [00:00<00:00, 167.43 examples/s]


{'tokens': ['John', 'works', 'at', 'Google'],
 'pos_tags': [1, 4, 2, 1],
 'chunk_tags': [4, 0, 3, 4]}

In [19]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [20]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    
    labels = []
    word_ids = tokenized_inputs.word_ids()
    
    previous_word_idx = None
    
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["pos_tags"][word_idx])  # POS tagging
        else:
            labels.append(-100)
        
        previous_word_idx = word_idx
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [21]:
tokenized_datasets = dataset.map(tokenize_and_align_labels)

tokenized_datasets["train"][0]

Map: 100%|██████████| 3/3 [00:00<00:00, 170.06 examples/s]


{'tokens': ['John', 'works', 'at', 'Google'],
 'pos_tags': [1, 4, 2, 1],
 'chunk_tags': [4, 0, 3, 4],
 'input_ids': [101, 2198, 2573, 2012, 8224, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1],
 'labels': [-100, 1, 4, 2, 1, -100]}

In [22]:
# Number of POS labels

num_labels = len(pos_labels)

print("Number of labels:", num_labels)

Number of labels: 7


In [23]:
# Load BERT model for token classification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=pos_id2label,
    label2id=pos_label2id
)

model.to(device)

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [24]:
# Handles dynamic padding

data_collator = DataCollatorForTokenClassification(tokenizer)

In [25]:
# Training configuration

training_args = TrainingArguments(
    output_dir="./results",
    
    num_train_epochs=3,   # small dataset → slightly more epochs
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    
    learning_rate=2e-5,
    
    logging_steps=10,
    
    do_train=True,
    do_eval=True
)

In [26]:
# Load seqeval metric

metric = evaluate.load("seqeval")

In [27]:
# Metrics function

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_preds = []
        curr_labels = []
        
        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_preds.append(pos_id2label[p_val])
                curr_labels.append(pos_id2label[l_val])
        
        true_predictions.append(curr_preds)
        true_labels.append(curr_labels)

    results = metric.compute(predictions=true_predictions, references=true_labels)
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [29]:
trainer.train()

{'train_runtime': '12.57', 'train_samples_per_second': '0.716', 'train_steps_per_second': '0.477', 'train_loss': '1.839', 'epoch': '3'}


TrainOutput(global_step=6, training_loss=1.8393839200337727, metrics={'train_runtime': 12.5715, 'train_samples_per_second': 0.716, 'train_steps_per_second': 0.477, 'train_loss': 1.8393839200337727, 'epoch': 3.0})

In [30]:
# Evaluate model

results = trainer.evaluate()

print("Evaluation Results:")
for key, value in results.items():
    print(f"{key}: {value}")

{'eval_loss': '1.588', 'eval_precision': '0.7273', 'eval_recall': '0.7273', 'eval_f1': '0.7273', 'eval_accuracy': '0.75', 'eval_runtime': '0.3415', 'eval_samples_per_second': '8.784', 'eval_steps_per_second': '5.856', 'epoch': '3'}
Evaluation Results:
eval_loss: 1.587873935699463
eval_precision: 0.7272727272727273
eval_recall: 0.7272727272727273
eval_f1: 0.7272727272727273
eval_accuracy: 0.75
eval_runtime: 0.3415
eval_samples_per_second: 8.784
eval_steps_per_second: 5.856
epoch: 3.0


In [31]:
# Create inference pipeline

nlp = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)

In [32]:
# Test sentence

sentence = "John works at Google in California"

output = nlp(sentence)

for token in output:
    print(token)

{'entity': 'VBZ', 'score': np.float32(0.19540703), 'index': 1, 'word': 'john', 'start': 0, 'end': 4}
{'entity': 'VBZ', 'score': np.float32(0.2842873), 'index': 2, 'word': 'works', 'start': 5, 'end': 10}
{'entity': 'NNP', 'score': np.float32(0.17629044), 'index': 3, 'word': 'at', 'start': 11, 'end': 13}
{'entity': 'VBZ', 'score': np.float32(0.20920603), 'index': 4, 'word': 'google', 'start': 14, 'end': 20}
{'entity': 'IN', 'score': np.float32(0.19017766), 'index': 5, 'word': 'in', 'start': 21, 'end': 23}
{'entity': 'NNP', 'score': np.float32(0.20766512), 'index': 6, 'word': 'california', 'start': 24, 'end': 34}


In [33]:
# Test sentence

sentence = "John works at Google in California"

output = nlp(sentence)

for token in output:
    print(token)

{'entity': 'VBZ', 'score': np.float32(0.19540703), 'index': 1, 'word': 'john', 'start': 0, 'end': 4}
{'entity': 'VBZ', 'score': np.float32(0.2842873), 'index': 2, 'word': 'works', 'start': 5, 'end': 10}
{'entity': 'NNP', 'score': np.float32(0.17629044), 'index': 3, 'word': 'at', 'start': 11, 'end': 13}
{'entity': 'VBZ', 'score': np.float32(0.20920603), 'index': 4, 'word': 'google', 'start': 14, 'end': 20}
{'entity': 'IN', 'score': np.float32(0.19017766), 'index': 5, 'word': 'in', 'start': 21, 'end': 23}
{'entity': 'NNP', 'score': np.float32(0.20766512), 'index': 6, 'word': 'california', 'start': 24, 'end': 34}


In [34]:
# Clean readable output

for token in output:
    print(f"{token['word']} → {token['entity']}")

john → VBZ
works → VBZ
at → NNP
google → VBZ
in → IN
california → NNP


### 📊 POS Tagging vs Chunking — Comparison

1. Part-of-Speech (POS) Tagging:  
Assigns grammatical labels to each word  
Examples:  
- Noun (NN)
- Verb (VB)
- Adjective (JJ)  
Focus: Word-level classification  
Complexity: Easier  

2. Chunking (Phrase Detection):  
Groups words into meaningful phrases  
Examples:  
- Noun Phrase (NP)  
- Verb Phrase (VP)  
Uses BIO tagging:  
- B → Beginning  
- I → Inside  
- O → Outside  
Focus: Phrase-level understanding  
Complexity: Higher than POS tagging  


3. 🧪 Observations  
- POS tagging is simpler because it labels individual words
- Chunking is more complex because it depends on context and grouping
- Transformer models like BERT perform well for both tasks due to contextual understanding
- Token classification requires careful handling of subwords and label alignment  



4. ⚠️ Challenges Faced
- Handling tokenization where one word splits into multiple subwords
- Aligning labels correctly using -100 for ignored tokens
- Managing compatibility issues with datasets library
- Training on CPU required optimization and smaller dataset usage    


5. ✅ Conclusion
- Successfully implemented token classification using BERT
- Performed POS tagging using transformer-based model
- Understood:
    - Tokenization and label alignment
    - Transformer fine-tuning for sequence labeling
    - Evaluation using sequence metrics (seqeval)
    - BERT demonstrates strong capability in understanding context at token level  


6. 🚀 Future Improvements  
- Train on larger datasets like CoNLL-2003
- Use GPU for faster training
- Extend to chunking model training separately
- Try advanced models like DistilBERT or RoBERTa